# Bank Marketing — Profit-Optimized Targeting (CatBoost)

This notebook implements **two decision policies** for a bank marketing call campaign:

1. **Profit-maximizing threshold (operations view)**  
   Call if `p >= t*`, where `t*` is chosen to maximize  
   `Profit = TP * revenue - (TP + FP) * cost`

2. **Fixed-budget targeting (campaign view)**  
   If you can only call a fixed fraction (Top **5% / 10% / 20%**), evaluate
   precision/recall and profit under that budget:  
   `Profit_topK = TP * revenue - K * cost`

We **learn the policy on OOF (out-of-fold) predictions** to reduce overfitting,
then apply the same policy on the test split (if labels exist).


In [ ]:
# ===== 0) (Optional) install =====
# Kaggle usually has catboost already. If not, uncomment:
# !pip -q install catboost

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss


In [ ]:
# ===== 1) Config =====
SEED = 42
SEP = ";"

REVENUE = 20.0  # TP revenue
COST = 1.0      # cost per call

TOP_PCTS = [0.05, 0.10, 0.20]
CV_SPLITS = 5

DROP_COLS = ["y", "duration", "age", "default", "day", "pdays"]  # final feature set


In [ ]:
# ===== 2) Load data (Kaggle-friendly) =====
def find_file_by_name(name: str):
    # Kaggle inputs live here:
    root = Path("/kaggle/input")
    if root.exists():
        hits = list(root.rglob(name))
        if len(hits) == 0:
            return None
        return hits[0]
    # fallback: local repo style
    for cand in [Path("data")/name, Path(name)]:
        if cand.exists():
            return cand
    return None

train_path = find_file_by_name("train.csv")
test_path  = find_file_by_name("test.csv")

print("train_path:", train_path)
print("test_path:", test_path)

assert train_path is not None, "train.csv not found. Put it in the Kaggle Dataset or /kaggle/input."
train = pd.read_csv(train_path, sep=SEP)
test = pd.read_csv(test_path,  sep=SEP) if test_path is not None else None

display(train.head())
print("train shape:", train.shape)
if test is not None:
    print("test shape :", test.shape)
print("columns:", list(train.columns))


## Quick sanity checks (EDA-light)
- Target balance
- AUC/PR baseline reference (positive rate)


In [ ]:
y_train = train["y"].map({"yes": 1, "no": 0}).astype(int)
pos_rate = float(y_train.mean())
print("Positive rate (train):", pos_rate)


## 3) Feature building (final set)

We intentionally drop:
- `duration` (leakage)
- and the final agreed drops: `age`, `default`, `day`, `pdays`  
and keep an engineered indicator:
- `never_contacted = 1(pdays == -1)`


In [ ]:
def build_features(df: pd.DataFrame):
    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore").copy()
    if "pdays" in df.columns:
        X["never_contacted"] = (df["pdays"] == -1).astype(int)
    else:
        X["never_contacted"] = 0

    cat_cols = X.select_dtypes(include="object").columns.tolist()
    for c in cat_cols:
        X[c] = X[c].astype(str).fillna("NA")
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    return X, cat_cols, cat_idx

X, cat_cols, cat_idx = build_features(train)
print("X shape:", X.shape)
print("cat cols:", cat_cols)


## 4) Helper metrics: Top-p% + Profit

- **Top-p%** is a fixed-budget campaign view
- **Profit threshold** is an operations view


In [ ]:
def top_k_summary(proba, y_true, pct_list=TOP_PCTS):
    y_true = pd.Series(y_true).reset_index(drop=True).astype(int)
    proba = np.asarray(proba)

    n = len(proba)
    total_pos = int(y_true.sum())
    rows = []
    for pct in pct_list:
        k = int(np.ceil(n * pct))
        idx = np.argsort(-proba)[:k]
        tp = int(y_true.iloc[idx].sum())
        fp = int(k - tp)
        precision = tp / k if k > 0 else np.nan
        recall = tp / total_pos if total_pos > 0 else np.nan
        rows.append({
            "top_pct": pct,
            "k_calls": k,
            "tp": tp,
            "fp": fp,
            "precision": precision,
            "recall": recall,
        })
    return pd.DataFrame(rows)

def profit_at_threshold(y_true, proba, thr, revenue=REVENUE, cost=COST):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba)
    pred = (proba >= thr).astype(int)
    tp = int(((pred==1)&(y_true==1)).sum())
    fp = int(((pred==1)&(y_true==0)).sum())
    fn = int(((pred==0)&(y_true==1)).sum())
    profit = tp*revenue - (tp+fp)*cost
    precision = tp/(tp+fp+1e-12)
    recall = tp/(tp+fn+1e-12)
    call_rate = float(pred.mean())
    return {"threshold": float(thr), "tp": tp, "fp": fp, "calls": int(tp+fp), "profit": float(profit),
            "precision": float(precision), "recall": float(recall), "call_rate": call_rate}

def best_threshold_by_profit(y_true, proba, revenue=REVENUE, cost=COST, grid=1001):
    thresholds = np.linspace(0.0, 1.0, grid)
    best = None
    best_profit = -1e18
    for thr in thresholds:
        stat = profit_at_threshold(y_true, proba, float(thr), revenue=revenue, cost=cost)
        if stat["profit"] > best_profit:
            best_profit = stat["profit"]
            best = stat
    return best

def profit_at_top_pct(y_true, proba, top_pct, revenue=REVENUE, cost=COST):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba)
    n = len(proba)
    k = int(np.ceil(n * float(top_pct)))
    idx = np.argsort(-proba)[:k]
    tp = int(y_true[idx].sum())
    fp = int(k - tp)
    profit = tp*revenue - k*cost
    precision = tp/k if k > 0 else np.nan
    recall = tp/(y_true.sum()+1e-12)
    return {"top_pct": float(top_pct), "k": int(k), "tp": tp, "fp": fp,
            "profit": float(profit), "precision": float(precision), "recall": float(recall)}


## 5) Main evaluation: OOF CV (policy is learned here)

We train CatBoost with StratifiedKFold and create **OOF probabilities**.
Then we:
- report ROC-AUC / PR-AUC (AP)
- report Top 5/10/20% summaries
- find profit-maximizing threshold `t*` on OOF
- (optional) isotonic calibration and compare


In [ ]:
# ===== 5) OOF CV =====
y = y_train.values
oof_proba = np.zeros(len(X), dtype=float)

skf = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
    va_pool = Pool(X_va, y_va, cat_features=cat_idx)

    model = CatBoostClassifier(
        iterations=830,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        verbose=0,
    )
    model.fit(tr_pool)
    oof_proba[va_idx] = model.predict_proba(va_pool)[:, 1]

oof_roc = roc_auc_score(y, oof_proba)
oof_pr  = average_precision_score(y, oof_proba)
print("OOF ROC-AUC:", oof_roc)
print("OOF PR-AUC (AP):", oof_pr)
print("OOF Lift over baseline:", oof_pr / (y.mean()+1e-12))

display(top_k_summary(oof_proba, y))

best_oof = best_threshold_by_profit(y, oof_proba, revenue=REVENUE, cost=COST, grid=1001)
print("\n[Best profit threshold on OOF]")
print(best_oof)

# Fixed-budget profit
oof_profit_topk = [profit_at_top_pct(y, oof_proba, p, REVENUE, COST) for p in TOP_PCTS]
display(pd.DataFrame(oof_profit_topk))


In [ ]:
# ===== 5-1) (Optional) Isotonic calibration on OOF =====
USE_ISOTONIC = True

best_oof_iso = None
iso_report = None

if USE_ISOTONIC:
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof_proba, y)
    oof_iso = iso.transform(oof_proba)

    iso_report = {
        "brier_raw": float(brier_score_loss(y, oof_proba)),
        "brier_iso": float(brier_score_loss(y, oof_iso)),
        "roc_raw": float(roc_auc_score(y, oof_proba)),
        "roc_iso": float(roc_auc_score(y, oof_iso)),
        "pr_raw": float(average_precision_score(y, oof_proba)),
        "pr_iso": float(average_precision_score(y, oof_iso)),
    }
    print("Isotonic report:", iso_report)

    best_oof_iso = best_threshold_by_profit(y, oof_iso, revenue=REVENUE, cost=COST, grid=1001)
    print("\n[Best profit threshold on OOF (Isotonic)]")
    print(best_oof_iso)


## 6) Train final model on full train + evaluate on test (if labels exist)

We apply **the same policy** learned on OOF:
- Profit-threshold policy (raw or isotonic)
- Fixed-budget policy Top 5/10/20%

If test is unlabeled (common in Kaggle), we still export call lists.


In [ ]:
# ===== 6) Train full model =====
train_pool_full = Pool(X, y, cat_features=cat_idx)

final_model = CatBoostClassifier(
    iterations=830,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    verbose=0,
)
final_model.fit(train_pool_full)

# ===== 6-1) Predict on test (if provided) =====
if test is None:
    print("No test.csv provided.")
else:
    X_test, _, _ = build_features(test)
    X_test = X_test.reindex(columns=X.columns)  # align columns

    test_pool = Pool(X_test, cat_features=cat_idx)
    proba_test = final_model.predict_proba(test_pool)[:, 1]

    # Determine which threshold to use for calling
    thr_theory = COST / REVENUE
    thr_profit = best_oof["threshold"]
    thr_profit_iso = best_oof_iso["threshold"] if best_oof_iso is not None else None

    print("thr_theory (cost/revenue):", thr_theory)
    print("thr_profit (OOF best):    ", thr_profit)
    if thr_profit_iso is not None:
        print("thr_profit_iso (OOF best):", thr_profit_iso)

    # If labels exist, evaluate
    if "y" in test.columns:
        y_test = test["y"].map({"yes": 1, "no": 0}).astype(int).values
        print("\n[Test metrics]")
        print("ROC-AUC:", roc_auc_score(y_test, proba_test))
        print("PR-AUC :", average_precision_score(y_test, proba_test))

        print("\n[Policy: Profit-threshold (OOF best)]")
        print(profit_at_threshold(y_test, proba_test, thr_profit, REVENUE, COST))

        if thr_profit_iso is not None:
            # apply isotonic calibrated threshold (calibration learned on OOF)
            proba_test_iso = iso.transform(proba_test)
            print("\n[Policy: Profit-threshold (Isotonic OOF best)]")
            print(profit_at_threshold(y_test, proba_test_iso, thr_profit_iso, REVENUE, COST))

        print("\n[Policy: Top-p% summaries]")
        display(top_k_summary(proba_test, y_test))
        display(pd.DataFrame([profit_at_top_pct(y_test, proba_test, p, REVENUE, COST) for p in TOP_PCTS]))

    # ===== Export call lists =====
    out = test.copy()
    out["pred_proba"] = proba_test
    out["call_flag_profit"] = (proba_test >= thr_profit).astype(int)
    out.to_csv("call_target_list_profit_threshold.csv", index=False)

    # Fixed budget lists
    for p in TOP_PCTS:
        k = int(np.ceil(len(proba_test) * p))
        top_idx = np.argsort(-proba_test)[:k]
        out_top = test.iloc[top_idx].copy()
        out_top["pred_proba"] = proba_test[top_idx]
        out_top.to_csv(f"call_target_list_top{int(p*100)}pct.csv", index=False)

    print("Saved: call_target_list_profit_threshold.csv and call_target_list_top{5,10,20}pct.csv")
